# LoRA fine-tuning of one LLM configuration

**To run a different experiment, change only `CONFIG` in cells 4 and 5.** Available:
`lora_allam7b_c2` (start here: the paper's best model), `lora_allam7b_c1`, `lora_allam7b_c3`,
`lora_llama3_8b_c1/c2/c4`, `lora_mistral7b_c1/c2` (in `configs/`).
LLaMA 3 needs a Hugging Face token (cell 3). **Settings:** GPU T4 x2, internet ON.

The code has never run on a GPU, so cell 4 (dry run, a few minutes) is essential.
I estimate 1-1.5 h per full run; the dry run will tell us the real speed.
Tip: for the long run use **Save Version -> Save & Run All** so it keeps going if you close the browser.

## 1. Get the code

In [ ]:
import os, glob, subprocess

# >>> EDIT THIS LINE: your GitHub username <<<
REPO_URL = "https://github.com/YOUR_USERNAME/Reproduction-of-domain-specific-Tunisian-Arabic-dialect-to-MSA-translation.git"

WORK = "/kaggle/working"
name = REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]
path = os.path.join(WORK, name)
if not os.path.exists(path):
    subprocess.run(["git", "clone", REPO_URL, path], check=True)
else:
    subprocess.run(["git", "-C", path, "checkout", "--", "."], check=True)
    subprocess.run(["git", "-C", path, "pull"], check=True)
hits = sorted(glob.glob(path + "/**/src/train_seq2seq.py", recursive=True), key=len)
assert hits, "src/train_seq2seq.py not found: check REPO_URL and that you pushed the files"
os.chdir(os.path.dirname(os.path.dirname(hits[0])))
print("working directory:", os.getcwd())
assert os.path.exists("src/summarize.py") and os.path.exists("src/train_lora.py"), "Old code on GitHub: git add . / commit / push the newest files, then re-run this cell"
print("code is up to date")

## 2. Install libraries (bitsandbytes is needed for 4-bit)

In [ ]:
!pip install -q "transformers>=4.46,<5" "peft>=0.12" bitsandbytes accelerate sacrebleu pyyaml sentencepiece protobuf

## 3. Hugging Face token (only needed for LLaMA 3; skip for ALLaM)
Add a Kaggle secret named HF_TOKEN (Add-ons -> Secrets), and accept the Llama 3 licence on huggingface.co.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret (fine for ALLaM/Mistral):", type(e).__name__)

## 4. DRY RUN (a few minutes). Check that 4-bit loading, training, validation BLEU and generation all work.

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

CONFIG = "configs/lora_allam7b_c2.yaml"
!python -m src.train_lora --config {CONFIG} --dry_run

**Send me the first 15 and last 40 lines of that output**, or the full error. Do not start cell 5 until the dry run succeeded.

## 5. FULL RUN (long)

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

CONFIG = "configs/lora_allam7b_c2.yaml"
!python -m src.train_lora --config {CONFIG}

## 6. SUMMARY - send me this table as text

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

!python -m src.summarize results/table8_lora

## 7. Package results (adapters are large and stay in checkpoints/, only results/ is zipped)

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

import shutil
!zip -qr results_lora.zip results
shutil.copy("results_lora.zip", "/kaggle/working/results_lora.zip")